### Read Files


In [2]:
using PowerModels
using JuMP,Gurobi, DataFrames, XLSX,Random,Distributions
# Specify the path to your MATPOWER file
file_path = "/Users/Patron/Documents/Switch_Instances/case118Blumsack.m"
file_path_excel = "/Users/Patron/Documents/Switch_Instances/RandomDemands.xlsx"

# Parse the MATPOWER case file
network = PowerModels.parse_file(file_path);

# Read the Excel data containing random parameters 
param_data = XLSX.readtable(file_path_excel, "118_15") |> DataFrame

center_data = [56 27 42 50 0 59 29 40 0 0 80 56 46 17 99 37 24 69 56 25 16 23 7 20 0 0 83 31 28 0 47 65 29 71 35 32 0 0 39 74 48 111 31 30 65 40 42 35 88 24 27 28 24 124 70 88 22 25 278 92 0 83 0 0 0 47 41 0 0 75 0 20 21 75 47 72 76 79 46 130 0 65 33 23 24 32 0 63 0 450 19 79 26 45 44 42 17 42 57 38 24 10 38 51 33 46 51 6 13 51 0 73 13 18 23 199 22 42];
Pd = 0.01.*collect(center_data[1,:]);


[warn | PowerModels]: this code only supports angmin values in -90 deg. to 90 deg., tightening the value on branch 1 from -360.0 to -60.0 deg.
[warn | PowerModels]: this code only supports angmax values in -90 deg. to 90 deg., tightening the value on branch 1 from 360.0 to 60.0 deg.
[warn | PowerModels]: this code only supports angmin values in -90 deg. to 90 deg., tightening the value on branch 54 from -360.0 to -60.0 deg.
[warn | PowerModels]: this code only supports angmax values in -90 deg. to 90 deg., tightening the value on branch 54 from 360.0 to 60.0 deg.
[warn | PowerModels]: this code only supports angmin values in -90 deg. to 90 deg., tightening the value on branch 101 from -360.0 to -60.0 deg.
[warn | PowerModels]: this code only supports angmax values in -90 deg. to 90 deg., tightening the value on branch 101 from 360.0 to 60.0 deg.
[warn | PowerModels]: this code only supports angmin values in -90 deg. to 90 deg., tightening the value on branch 41 from -360.0 to -60.0 deg

### Parameters

In [3]:
function parameters(network)
    #Sets
    B = network["bus"]
    L = network["branch"]
    G = network["gen"]

    #Parameters
    B_ij = zeros(length(L)) # Suspectance valuesL
    fbar_ij = zeros(length(L)) # Upper bound on the load
    p_max = zeros(length(B)) #Max power ouput
    p_min = zeros(length(B)) #Min power output
    c = zeros(length(B)) #Linear Cost coefficient with zero intercept

    for (branch_id,branch) in L
        B_ij[parse(Int,branch_id)] = -branch["br_x"]/(branch["br_r"]^2 + branch["br_x"]^2)  # B_ij = -reactance/(resistance^2+reactance^2)
        fbar_ij[parse(Int,branch_id)] = branch["rate_a"]   #f_max = rate A
    end

    for (gen_id,generator) in G
        c[generator["gen_bus"]] = generator["cost"][1]    #generator linear cost funcion slope with 0 intercept
        p_max[generator["gen_bus"]] = generator["pmax"]     # max power 
        p_min[generator["gen_bus"]] = generator["pmin"]     # min power
    end

    return B_ij,fbar_ij,p_max,p_min,c
end

parameters (generic function with 1 method)

### Random z vector

In [11]:
using Random, Distributions

"""
    random_z(network; p=0.5, rng=GLOBAL_RNG)

Return a vector `z` of length `nb = length(network["bus"])` where
- for each generator bus `i` in `network["gen"]`, `z[i]` ~ Bernoulli(p) ∈ {0,1}
- for all other buses `j`, `z[j] == 0`
"""
function random_z(network; p=0.5)
    nb = length(network["bus"])
    z  = zeros(Int, nb)                       # start all zeros
    gens = [ gen["gen_bus"] for gen in values(network["gen"]) ]
    for i in gens
        z[i] = rand(Bernoulli(p))        # 1 w.p. p, else 0
    end
    return z
end
Random.seed!(1234)
z_sample = random_z(network; p=0.9)
sum(z_sample)


16

### Offline Calculations

In [13]:
function offline_calc(network, Pd, z)
    #Initialize the model
    model = Model(Gurobi.Optimizer)
    MOI.set(model, MOI.TimeLimitSec(), 30.0)

    #Sets
    B = network["bus"]
    L = network["branch"]
    G = network["gen"]

    #Decision Variables
    @variable(model, pg[1: length(B)]   >= 0)    # generation
    @variable(model, theta[1: length(B)])  # Voltage angle decision variable
    @variable(model, f[1:length(L)]) # Current flow decision variable 
    @variable(model, x[1:length(L)],Bin) # Switching decision variable
    @variable(model, unmet[1:length(B)]>=0)
    Penalty = 10000

    nb = length(B)    # Number of buses
    
    B_ij,f_ij,p_max,p_min,c_i = parameters(network)

    # Objective
    @objective(model, Min,
        sum(c_i[i]*pg[i] for i in 1:length(B)) +
        Penalty * sum(unmet)
    )
    
    # Constraints

    # 1) generation limits, gated by z
    @constraint(model, [i=1:length(B)],
        pg[i] >= p_min[i] * z[i],
    )
    @constraint(model, [i=1:length(B)],
        pg[i] <= p_max[i] * z[i],
    )

    # 2) nodal balance (original equality) → split into two inequalities
    for (bus_id, _) in B
        i = parse(Int, bus_id)
        # collect outgoing and incoming flows
        sum_out = sum( f[l] for (branch_id, branch) in L if branch["f_bus"] == i
                            for l = [parse(Int, branch_id)]; init=0 )
        sum_in  = sum( f[l] for (branch_id, branch) in L if branch["t_bus"] == i
                            for l = [parse(Int, branch_id)]; init=0 )

        #   (pg - Pd) - (sum_out - sum_in) + unmet[2i-1] ≥ 0
        @constraint(model,
            pg[i] - Pd[i] - (sum_out - sum_in) + unmet[i] == 0
        )
    end
    #@constraint(model, [i = 1:length(L)], x[i] <= 1)
    for (branch_id,branch) in L
        branch_index = parse(Int,branch_id)
        f_bus = branch["f_bus"]
        t_bus = branch["t_bus"]
        M_ij = -12.6*B_ij[branch_index]
        @constraint(model,-f[branch_index] - f_ij[branch_index]*x[branch_index]<=0)
        @constraint(model,f[branch_index] - f_ij[branch_index]*x[branch_index]<=0)
    
        # Linearizing them
        @constraint(model,B_ij[branch_index]*(theta[f_bus] - theta[t_bus]) - M_ij*(1-x[branch_index])-f[branch_index]<=0)
        @constraint(model,-B_ij[branch_index]*(theta[f_bus] - theta[t_bus]) - M_ij*(1-x[branch_index])+f[branch_index]<=0)

    end
    
   
    optimize!(model)
    

    if termination_status(model) == MOI.OPTIMAL
        result = value.(x)
        return objective_value(model),collect(value.(x)), collect(value.(pg))
    else
        return -1,[],[]
    end
end

offline_calc (generic function with 1 method)

### Scenario Generation

In [15]:
function generate_scenarios(Pd,numscenarios)
    samples = zeros(numscenarios,length(Pd))
    # Global min
    #global_min = minimum(Pd)
    #global_max = maximum(Pd)
    
    for scenario in 1:numscenarios
        #global_value = round(rand(Uniform(global_min, global_max)),digits=0)  # Global value
        global_factor = rand(Uniform(0.8, 1.2)) #Global factor
        for i in 1:length(Pd)
            center = Pd[i]
            if center == 0
                #min_val = 0
                #max_val = 1  # Default range for zero-center buses
                local_factor_i = rand(Uniform(0,1))
            else
            #min_val = max(0, center * 0.80)  # 20% below center
            #max_val = center * 1.2         # 20% above center
                local_factor_i = 1 + rand(Uniform(-0.1, 0.1))  # ±20% local
            end
            bus_demand_i = Pd[i] * 1 * local_factor_i
            samples[scenario,i] = bus_demand_i
            #samples[scenario,i]= round(rand(Uniform(min_val, max_val))*(1+(global_value*0.0)/global_max),digits=0)
        end
    end
    return samples
end  

generate_scenarios (generic function with 1 method)

### Sampling

In [44]:
function sampling_for_offline_calc(network,Pd,z,total_scenarios; capacity_factor::Real = 5.0 )
    # 1) unpack network parameters
    B_ij,fbar_ij,p_max_base,p_min_base,c = parameters(network)

    # index sets
    buses    = sort(parse.(Int, collect(keys(network["bus"]))))
    branches = sort(parse.(Int, collect(keys(network["branch"]))))
    nb       = length(buses)
    nl       = length(branches)

    # 2) pre-allocate storage
    demand_samples = zeros(Float64,total_scenarios, nb)
    x_solutions    = zeros(total_scenarios, nl)
    pg_solutions   = zeros(total_scenarios, nb  )

    # 3) sampling loop
    found = 0
    while found < total_scenarios
        # generate one draw of demand & cost
        ds = generate_scenarios(Pd, 1)
        #print(ds)
        #d = vec(ds[1, :])   # demand scenario

        # make *local* scaled copies of your capacities
        p_max = p_max_base .* capacity_factor
        p_min = p_min_base .* capacity_factor

        # random z
        z1 = random_z(network; p=0.9)

        # try to solve
        obj,x,pg = offline_calc(network,generate_scenarios(Pd, 1), z1)

        # if we get here, it solved to optimality
        if obj!=-1
            found += 1
            demand_samples[found, :] = ds
            x_solutions[found, :] = x
            pg_solutions[found, :] = pg

        else
            println("Scenario is infeasible, trying again")
            # just loop and sample again
        end
    end

    return demand_samples, x_solutions, pg_solutions
end
demand_samples,x_solutions, pg_solutions = sampling_for_offline_calc(network,Pd,z_sample,15)

Set parameter Username
Set parameter LicenseID to value 2686057
Academic license - for non-commercial use only - expires 2026-07-09
Set parameter TimeLimit to value 30
Set parameter TimeLimit to value 30
Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (mac64[arm] - Darwin 23.6.0 23H626)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  30

Optimize a model with 1098 rows, 726 columns and 3076 nonzeros
Model fingerprint: 0x0ff5bb90
Variable types: 540 continuous, 186 integer (186 binary)
Coefficient statistics:
  Matrix range     [1e+00, 3e+03]
  Objective range  [2e+01, 1e+04]
  Bounds range     [0e+00, 0e+00]
  RHS range        [6e-02, 3e+03]
Found heuristic solution: objective 531541.22489
Presolve removed 301 rows and 258 columns
Presolve time: 0.00s
Presolved: 797 rows, 468 columns, 2424 nonzeros
Found heuristic solution: objective 514995.51573
Variable types: 295 continuous, 173 integer (173 binary

([0.5819789903638216 0.2572613633430705 … 0.23286946238211953 0.41092594689209516; 0.6101889708251621 0.2946648446717562 … 0.19853299012101241 0.42667173513878204; … ; 0.6149607304360825 0.2682103853092441 … 0.2140428052902032 0.4306044032813082; 0.6097700708716396 0.26301156049060503 … 0.20723089623119442 0.45449294260436546], [1.0 -0.0 … 1.0 -0.0; 1.0 0.0 … 1.0 0.0; … ; 1.0 -0.0 … 1.0 -0.0; -0.0 1.0 … 1.0 -0.0], [0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0])

In [17]:
x_solutions

10×186 Matrix{Float64}:
  1.0  -0.0   1.0   1.0   1.0   1.0  …   1.0   1.0  1.0  1.0   1.0  -0.0
  1.0   1.0   1.0   1.0  -0.0  -0.0      1.0   1.0  1.0  1.0  -0.0   1.0
 -0.0   1.0  -0.0   1.0  -0.0  -0.0      1.0   1.0  1.0  1.0   1.0  -0.0
 -0.0  -0.0  -0.0  -0.0   1.0   1.0     -0.0   1.0  1.0  1.0   1.0   1.0
  1.0   1.0   1.0  -0.0   1.0   1.0     -0.0   1.0  1.0  1.0   1.0   1.0
  1.0  -0.0   1.0   1.0   1.0   1.0  …   1.0  -0.0  1.0  1.0   1.0   1.0
 -0.0   1.0   1.0   1.0   1.0   1.0      0.0   1.0  1.0  1.0   1.0   1.0
 -0.0   1.0   1.0   1.0   1.0   1.0      1.0   1.0  1.0  1.0   1.0   1.0
  0.0   1.0   1.0   1.0   1.0   1.0      1.0   1.0  1.0  1.0   0.0   0.0
  1.0   1.0   1.0   1.0   1.0   1.0      1.0  -0.0  1.0  1.0   1.0   1.0

In [45]:
function n_unique_rows(A::AbstractMatrix)
    # turn every row into an immutable tuple
    rows_as_tuples = (Tuple(view(A, i, :)) for i in axes(A,1))
    uniq = unique(rows_as_tuples)     # removes duplicates
    return length(uniq)
end
n_unique_rows(x_solutions)

15

### Online samples

In [19]:
online_samples = generate_scenarios(Pd,200);
num_samples = 200
nb          = length(network["bus"])
Z           = zeros(Int, num_samples, nb)

for s in 1:num_samples
    Z[s, :] = random_z(network; p=0.9)
end


### Decoupling continuous and binary decision variable

In [20]:
function solve_for_continuous_var(network,sample,z, solved_x)
    #Initialize the model
    model = Model(Gurobi.Optimizer)
    MOI.set(model, MOI.TimeLimitSec(), 300.0)

    #Sets
    B = network["bus"]
    L = network["branch"]
    G = network["gen"]

    #Decision Variables
    @variable(model, pg[1: length(B)]   >= 0)    # generation
    @variable(model, theta[1: length(B)])  # Voltage angle decision variable
    @variable(model, f[1:length(L)]) # Current flow decision variable 
    @variable(model, x[1:length(L)],Bin) # Switching decision variable
    @variable(model, unmet[1:length(B)]>=0)
    Penalty = 10000

    B_ij,f_ij,p_max,p_min,c_i = parameters(network)

    # Objective
    @objective(model, Min,
        sum(c_i[i]*pg[i] for i in 1:length(B)) +
        Penalty * sum(unmet)
    )

    # Constraints

    # 1) generation limits, gated by z
    @constraint(model, [i=1:length(B)],
        pg[i] >= p_min[i] * z[i],
    )
    @constraint(model, [i=1:length(B)],
        pg[i] <= p_max[i] * z[i],
    )

    # 2) nodal balance (original equality) → split into two inequalities
    for (bus_id, _) in B
        i = parse(Int, bus_id)
        # collect outgoing and incoming flows
        sum_out = sum( f[l] for (branch_id, branch) in L if branch["f_bus"] == i
                            for l = [parse(Int, branch_id)]; init=0 )
        sum_in  = sum( f[l] for (branch_id, branch) in L if branch["t_bus"] == i
                            for l = [parse(Int, branch_id)]; init=0 )

        #   (pg - Pd) - (sum_out - sum_in) + unmet[2i-1] ≥ 0
        @constraint(model,
            pg[i] - sample[i] - (sum_out - sum_in) + unmet[i] == 0
        )
    end

    for (branch_id,branch) in L
        branch_index = parse(Int,branch_id)
        f_bus = branch["f_bus"]
        t_bus = branch["t_bus"]
        M_ij = -12.6*B_ij[branch_index]
        @constraint(model,-f[branch_index] - f_ij[branch_index]*x[branch_index]<=0)
        @constraint(model,f[branch_index] - f_ij[branch_index]*x[branch_index]<=0)
    
        # Linearizing them
        @constraint(model,B_ij[branch_index]*(theta[f_bus] - theta[t_bus]) - M_ij*(1-x[branch_index])-f[branch_index]<=0)
        @constraint(model,-B_ij[branch_index]*(theta[f_bus] - theta[t_bus]) - M_ij*(1-x[branch_index])+f[branch_index]<=0)

    end
    # 4) forcing binary variable to take a fixed value
    @constraint(model, x == solved_x )
    
    optimize!(model)
    # Check solver status
    if termination_status(model) == MOI.OPTIMAL
        return objective_value(model),collect(value.(pg))
    else
        return -1,[]
    end
end 

function Calculations_For_Extensive_Form(network,online_samples,Z,x_chosen)
    total_scenarios = size(online_samples,1)
    total_solutions = size(x_chosen,1)
    obj_value = zeros(total_scenarios,total_solutions)
    for i in 1:total_scenarios
        sample = online_samples[i,:]
        z = Z[i,:]
        for j in 1:total_solutions
            obj_value[i,j], _ = solve_for_continuous_var(network,sample,z, x_chosen[j,:])
        end
    end
    return obj_value
end


Calculations_For_Extensive_Form (generic function with 1 method)

In [21]:
obj_matrix = Calculations_For_Extensive_Form(network,online_samples,Z,x_solutions)

Set parameter Username
Set parameter LicenseID to value 2686057
Academic license - for non-commercial use only - expires 2026-07-09
Set parameter TimeLimit to value 300
Set parameter TimeLimit to value 300
Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (mac64[arm] - Darwin 23.6.0 23H626)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  300

Optimize a model with 1284 rows, 726 columns and 3262 nonzeros
Model fingerprint: 0x9b67dc0f
Variable types: 540 continuous, 186 integer (186 binary)
Coefficient statistics:
  Matrix range     [1e+00, 3e+03]
  Objective range  [2e+01, 1e+04]
  Bounds range     [0e+00, 0e+00]
  RHS range        [6e-02, 3e+03]
Found heuristic solution: objective 526021.05653
Presolve removed 1223 rows and 653 columns
Presolve time: 0.00s
Presolved: 61 rows, 73 columns, 211 nonzeros
Found heuristic solution: objective 411028.09534
Variable types: 73 continuous, 0 integer (0 binary)

R

200×10 Matrix{Float64}:
 89420.2             1.21914e5  …  84617.5        82941.4
 58889.5         84070.9           43732.9        51621.8
 48101.6         36659.3            3104.33        3215.29
 53861.8         28372.4           11183.4         8439.69
 45038.9         23164.2            3478.9         3637.62
 77982.4         66537.1        …  58835.2        54023.5
 41186.8         32159.0            4517.2         3958.26
 57003.3         68010.4           28442.8        29716.4
 44876.1         35044.5            4044.65        4037.75
 84145.9             1.03423e5     74815.3        89735.9
     ⋮                          ⋱                 
 41302.0         25416.9            3074.55        3119.81
 61189.9         94487.9           53215.4        53215.4
 77879.2        103190.0           61917.5        61917.5
 53410.5         52946.2           19181.5        29275.2
 47310.9         42100.3        …   4286.96        3923.57
     1.17447e5       1.45499e5         1.16521e5

In [40]:
function Reformulated_Extensive_Form(S,l,K,obj_value)
    # Initialize model
    model = Model(Gurobi.Optimizer)
    MOI.set(model, MOI.TimeLimitSec(), 30.0)

    #Total scenarios

    # Decision Variables
    @variable(model, z[1:l], Bin)
    @variable(model, w[1:l , 1:S], Bin)

    # Constraints
    for scenario in 1:S
        for solution in 1:l
            if obj_value[scenario,solution] == -1
                @constraint(model, w[solution,scenario] == 0)   ## weights of infeasible scenario/first stage decision pair is 0
            end
        end
    end
            
        
    for j in 1:S
        @constraint(model, [i in 1:l], w[i,j] <= z[i])
        @constraint(model, sum(w[i,j] for i in 1:l) == 1)
    end
    @constraint(model, sum(z[i] for i in 1:l) == K)

    # Objective
    objective = 0.0
    for scenario in 1:S
        for solution in 1:l
            objective += w[solution,scenario]*obj_value[scenario,solution]
        end
    end

    # Define the objective function
    @objective(model, Min, objective/S)

    # Solve the model
    #set_silent(model)
    optimize!(model)
    # Check solver status

    if termination_status(model) == MOI.OPTIMAL
        return objective_value(model),value.(z)
    else
        return -1,[]
    end    
end 
Solution,z=Reformulated_Extensive_Form(200,10,3,obj_matrix)

Set parameter Username
Set parameter LicenseID to value 2686057
Academic license - for non-commercial use only - expires 2026-07-09
Set parameter TimeLimit to value 30
Set parameter TimeLimit to value 30
Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (mac64[arm] - Darwin 23.6.0 23H626)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  30

Optimize a model with 2201 rows, 2010 columns and 6010 nonzeros
Model fingerprint: 0x452ac1fe
Variable types: 0 continuous, 2010 integer (2010 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+01, 1e+03]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 3e+00]
Presolve time: 0.00s
Presolved: 2201 rows, 2010 columns, 6010 nonzeros
Variable types: 0 continuous, 2010 integer (2010 binary)
Found heuristic solution: objective 51072.067312

Root relaxation: objective 3.212402e+04, 522 iterations, 0.01 seconds (0.01 work unit

(32124.024983504303, [-0.0, -0.0, -0.0, -0.0, 1.0, 1.0, -0.0, 1.0, 0.0, 0.0])

### Test sample

In [24]:
test_sample = generate_scenarios(Pd,20);
num_samples = 20
nb          = length(network["bus"])
Z_test           = zeros(Int, num_samples, nb)

for s in 1:num_samples
    Z_test[s, :] = random_z(network; p=0.8)
end

### Offline Calculation on Test Data

In [25]:
function OfflineTest(network,test_sample,Z_test)
    num_rows = size(test_sample, 1)
    offline_cost= zeros(num_rows)

    #Offline calculations for the test data
    total_cost = 0.0
    count = 0
    for i in 1:num_rows
        obj_value,_,_ = offline_calc(network,test_sample[i,:],Z_test[i,:])
        offline_cost[i] = obj_value
        if obj_value != -1
            total_cost += obj_value
            count += 1
        end
    end
    total_offline_cost = total_cost/count
    return total_offline_cost,offline_cost
end
original_cost,offline_cost_list = OfflineTest(network,test_sample,Z_test);

Set parameter Username
Set parameter LicenseID to value 2686057
Academic license - for non-commercial use only - expires 2026-07-09
Set parameter TimeLimit to value 30
Set parameter TimeLimit to value 30
Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (mac64[arm] - Darwin 23.6.0 23H626)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  30

Optimize a model with 1098 rows, 726 columns and 3076 nonzeros
Model fingerprint: 0x8b5fac74
Variable types: 540 continuous, 186 integer (186 binary)
Coefficient statistics:
  Matrix range     [1e+00, 3e+03]
  Objective range  [2e+01, 1e+04]
  Bounds range     [0e+00, 0e+00]
  RHS range        [7e-02, 3e+03]
Found heuristic solution: objective 532000.63744
Presolve removed 301 rows and 258 columns
Presolve time: 0.00s
Presolved: 797 rows, 468 columns, 2424 nonzeros
Found heuristic solution: objective 510157.13035
Variable types: 295 continuous, 173 integer (173 binary

### Second Stage Evaluation

In [26]:
function Second_Stage_Cost(network,sample,z_gen, x_chosen,z)
    #Initialize the model
    model = Model(Gurobi.Optimizer)
    MOI.set(model, MOI.TimeLimitSec(), 30.0)

    #Sets
    B = network["bus"]
    L = network["branch"]
    G = network["gen"]

    #Decision Variables
    @variable(model, pg[1: length(B)]   >= 0)    # generation
    @variable(model, theta[1: length(B)])  # Voltage angle decision variable
    @variable(model, f[1:length(L)]) # Current flow decision variable 
    @variable(model, x[1:length(L)],Bin) # Switching decision variable
    @variable(model, unmet[1:length(B)]>=0)
    @variable(model, w[1:length(z)], Bin)
    Penalty = 10000

    nb = length(B)    # Number of buses
    
    B_ij,f_ij,p_max,p_min,c_i = parameters(network)

    # Objective
    @objective(model, Min,
        sum(c_i[i]*pg[i] for i in 1:length(B)) +
        Penalty * sum(unmet)
    )
    
    # Constraints

    # 1) generation limits, gated by z_gen
    @constraint(model, [i=1:length(B)],
        pg[i] >= p_min[i] * z_gen[i],
    )
    @constraint(model, [i=1:length(B)],
        pg[i] <= p_max[i] * z_gen[i],
    )
    # 2) nodal balance (original equality) → split into two inequalities
    for (bus_id, _) in B
        i = parse(Int, bus_id)
        # collect outgoing and incoming flows
        sum_out = sum( f[l] for (branch_id, branch) in L if branch["f_bus"] == i
                            for l = [parse(Int, branch_id)]; init=0 )
        sum_in  = sum( f[l] for (branch_id, branch) in L if branch["t_bus"] == i
                            for l = [parse(Int, branch_id)]; init=0 )

        #   (pg - Pd) - (sum_out - sum_in) + unmet[2i-1] ≥ 0
        @constraint(model,
            pg[i] - Pd[i] - (sum_out - sum_in) + unmet[i] == 0
        )
    end
    #@constraint(model, [i = 1:length(L)], x[i] <= 1)
    for (branch_id,branch) in L
        branch_index = parse(Int,branch_id)
        f_bus = branch["f_bus"]
        t_bus = branch["t_bus"]
        M_ij = -12.6*B_ij[branch_index]
        @constraint(model,-f[branch_index] - f_ij[branch_index]*x[branch_index]<=0)
        @constraint(model,f[branch_index] - f_ij[branch_index]*x[branch_index]<=0)
    
        # Linearizing them
        @constraint(model,B_ij[branch_index]*(theta[f_bus] - theta[t_bus]) - M_ij*(1-x[branch_index])-f[branch_index]<=0)
        @constraint(model,-B_ij[branch_index]*(theta[f_bus] - theta[t_bus]) - M_ij*(1-x[branch_index])+f[branch_index]<=0)

    end
    @constraint(model, [i in 1:length(z)], w[i] <= z[i])
    @constraint(model, sum(w[i] for i in 1:length(z)) == 1)
    @constraint(model, x == sum(w[i] * x_chosen[i,:] for i in 1:length(z)))
   
    optimize!(model)

    # Solve the model
    set_silent(model)
    optimize!(model)

    # Check solver status

    if termination_status(model) == MOI.OPTIMAL
        return objective_value(model),value.(w)
    else
        #println("Solver did not find an optimal solution.")
        return -1,[]
    end
end  

Second_Stage_Cost (generic function with 1 method)

In [27]:
z

10-element Vector{Float64}:
  1.0
 -0.0
 -0.0
 -0.0
  1.0
  1.0
 -0.0
  1.0
  0.0
  1.0

### Evaluation

In [41]:
function Evaluation(network,test_sample,Z_test, offline_cost_list,x_chosen,z)
    num_rows = size(test_sample, 1)
    cost = zeros(num_rows)
    online_list = zeros(num_rows)
    ### Second stage cost
    model_cost = 0.0
    count = 0
    for index in 1:num_rows
        if offline_cost_list[index] != -1
            model_obj_value,_ = Second_Stage_Cost(network,test_sample[index,:],Z_test[index,:],x_chosen,z)
            online_list[index] = model_obj_value
            if model_obj_value != -1
                model_cost += model_obj_value
                count += 1
            end
        else
            online_list[index] = -1
        end
    end
    total_model_cost = model_cost / count

    return total_model_cost, online_list
end 
model_cost, model_cost_list = Evaluation(network,test_sample,Z_test,offline_cost_list,x_solutions,z);

Set parameter Username
Set parameter LicenseID to value 2686057
Academic license - for non-commercial use only - expires 2026-07-09
Set parameter TimeLimit to value 30
Set parameter TimeLimit to value 30
Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (mac64[arm] - Darwin 23.6.0 23H626)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  30

Optimize a model with 1295 rows, 736 columns and 4564 nonzeros
Model fingerprint: 0x754a67b4
Variable types: 540 continuous, 196 integer (196 binary)
Coefficient statistics:
  Matrix range     [1e+00, 3e+03]
  Objective range  [2e+01, 1e+04]
  Bounds range     [0e+00, 0e+00]
  RHS range        [6e-02, 3e+03]
Presolve removed 768 rows and 483 columns
Presolve time: 0.01s
Presolved: 527 rows, 253 columns, 1746 nonzeros
Variable types: 250 continuous, 3 integer (3 binary)
Found heuristic solution: objective 61742.548940

Root relaxation: objective 4.402973e+04, 365 itera

In [29]:
original_cost

78535.10250021891

In [42]:
model_cost

79323.81639387322